# 🩺 Module 11 Assignment — Diabetes Prediction Model

**Dataset:** Pima Indians Diabetes Database  
**Source:** [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)  
**Objective:** Train and evaluate ML classification models to predict whether a patient has diabetes.

---
### 📋 Table of Contents
1. Install & Import Libraries
2. Data Loading & Exploration
3. Data Preprocessing
4. Model Training
5. Model Evaluation & ROC Curves
6. Best Model Selection & Save


---
## 1️⃣ Install & Import Libraries

In [ ]:
# Install any missing packages (Colab usually has these pre-installed)
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & model selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    classification_report, confusion_matrix
)

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('✅ All libraries imported successfully!')

---
## 2️⃣ Data Loading & Exploration

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# Option A: load directly from URL (no Kaggle API needed)
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

col_names = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

df = pd.read_csv(url, names=col_names)
print('✅ Dataset loaded successfully!')
print(f'Shape: {df.shape}')

In [ ]:
# ── First few rows ────────────────────────────────────────────────────────────
print('📌 First 5 rows:')
df.head()

In [ ]:
# ── Dataset shape ─────────────────────────────────────────────────────────────
print(f'📐 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
print('🔍 Missing Values per Column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# ── Basic statistics ──────────────────────────────────────────────────────────
print('📊 Basic Statistics:')
df.describe()

In [ ]:
# ── Data types & info ─────────────────────────────────────────────────────────
print('🗂 Dataset Info:')
df.info()

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
print('🎯 Target Class Distribution:')
print(df['Outcome'].value_counts())
print(f'\nDiabetes Positive: {df["Outcome"].sum()} ({df["Outcome"].mean()*100:.1f}%)')
print(f'Diabetes Negative: {(df["Outcome"]==0).sum()} ({(1-df["Outcome"].mean())*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(x='Outcome', data=df, palette=['#2ecc71','#e74c3c'], ax=axes[0])
axes[0].set_title('Class Distribution (0=No Diabetes, 1=Diabetes)', fontsize=13)
axes[0].set_xlabel('Outcome')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(
    df['Outcome'].value_counts(),
    labels=['No Diabetes (0)', 'Diabetes (1)'],
    colors=['#2ecc71','#e74c3c'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05)
)
axes[1].set_title('Outcome Proportion', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature distributions ─────────────────────────────────────────────────────
features = df.columns[:-1].tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.histplot(data=df, x=feat, hue='Outcome', kde=True,
                 palette=['#2ecc71','#e74c3c'], ax=axes[i], alpha=0.6)
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Outcome', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Zero-value check (biologically implausible zeros) ─────────────────────────
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('⚠️  Zero values in biologically implausible columns (treated as missing):')
for col in zero_cols:
    zeros = (df[col] == 0).sum()
    print(f'  {col}: {zeros} zeros ({zeros/len(df)*100:.1f}%)')

---
## 3️⃣ Data Preprocessing

In [ ]:
# ── Replace biologically implausible zeros with column median ─────────────────
df_clean = df.copy()

for col in zero_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)

print('✅ Zeros replaced with column medians for:', zero_cols)
df_clean.describe()

In [ ]:
# ── Separate features (X) and target (y) ─────────────────────────────────────
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

print(f'✅ Features (X) shape : {X.shape}')
print(f'✅ Target  (y) shape  : {y.shape}')
print(f'\nFeature columns: {X.columns.tolist()}')

In [ ]:
# ── Train / Test Split (80% / 20%) ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'✅ Training set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'✅ Testing set   : {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# ── Feature Scaling (StandardScaler) ─────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # transform test (no fit!)

print('✅ Feature scaling applied (StandardScaler)')
print(f'  Mean of scaled train: {X_train_scaled.mean():.4f}  (≈ 0)')
print(f'  Std  of scaled train: {X_train_scaled.std():.4f}  (≈ 1)')

---
## 4️⃣ Model Training

We train **5 classification models**:
1. Logistic Regression
2. Random Forest
3. Support Vector Machine (SVM)
4. Decision Tree
5. K-Nearest Neighbors (KNN)

In [ ]:
# ── Define models ─────────────────────────────────────────────────────────────
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM'                 : SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7)
}

# ── Train all models ──────────────────────────────────────────────────────────
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f'✅ {name:25s} — trained!')

---
## 5️⃣ Model Evaluation & ROC Curves

In [ ]:
# ── Evaluate all models ───────────────────────────────────────────────────────
results = []

for name, model in trained_models.items():
    y_pred      = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

    row = {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1 Score' : f1_score(y_test, y_pred),
        'ROC-AUC'  : roc_auc_score(y_test, y_pred_prob)
    }
    results.append(row)

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print('📊 Model Evaluation Metrics:')
results_df.style.highlight_max(color='lightgreen').format('{:.4f}')

In [ ]:
# ── ROC Curve — all models on one plot ────────────────────────────────────────
plt.figure(figsize=(10, 7))

colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

for (name, model), color in zip(trained_models.items(), colors):
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    auc_score   = roc_auc_score(y_test, y_pred_prob)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.50)')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')

plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curve — All Models Comparison', fontsize=15)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curve saved as roc_curves.png')

In [ ]:
# ── Bar chart — metric comparison ─────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5), sharey=False)

palette = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics):
    sorted_df = results_df.sort_values(metric, ascending=True)
    axes[i].barh(sorted_df.index, sorted_df[metric],
                 color=palette, edgecolor='white')
    axes[i].set_xlim(0.5, 1.0)
    axes[i].set_title(metric, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Score')
    for j, v in enumerate(sorted_df[metric]):
        axes[i].text(v + 0.003, j, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('📊 Model Performance Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison chart saved as model_comparison.png')

In [ ]:
# ── Confusion matrices for all models ────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No DM','DM'], yticklabels=['No DM','DM'],
                cbar=False, linewidths=1)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed classification report for best model (by ROC-AUC) ───────────────
best_model_name = results_df['ROC-AUC'].idxmax()
best_model      = trained_models[best_model_name]
y_pred_best     = best_model.predict(X_test_scaled)

print(f'🏆 Best Model (by ROC-AUC): {best_model_name}')
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=['No Diabetes', 'Diabetes']))

---
## 6️⃣ Best Model Selection & Save

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 65)
print('           FINAL MODEL COMPARISON SUMMARY')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)
print(f'\n🏆 BEST MODEL  : {best_model_name}')
print(f'   ROC-AUC     : {results_df.loc[best_model_name, "ROC-AUC"]:.4f}')
print(f'   Accuracy    : {results_df.loc[best_model_name, "Accuracy"]:.4f}')
print(f'   F1 Score    : {results_df.loc[best_model_name, "F1 Score"]:.4f}')

In [ ]:
# ── Save best model using joblib ──────────────────────────────────────────────
joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(scaler,     'diabetes_scaler.pkl')   # save scaler too!

print(f'✅ Best model saved  → diabetes_model.pkl')
print(f'✅ Scaler saved      → diabetes_scaler.pkl')

In [ ]:
# ── Verify: reload and predict ────────────────────────────────────────────────
loaded_model  = joblib.load('diabetes_model.pkl')
loaded_scaler = joblib.load('diabetes_scaler.pkl')

# Sample prediction (first test row)
sample = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)
pred = loaded_model.predict(sample_scaled)[0]
prob = loaded_model.predict_proba(sample_scaled)[0][1]

print('🔁 Model reload verification:')
print(f'   Sample features : {sample.values.tolist()}')
print(f'   Prediction      : {"Diabetes" if pred == 1 else "No Diabetes"} (class {pred})')
print(f'   Probability     : {prob:.4f}')
print(f'   Actual label    : {y_test.iloc[0]}')
print('\n✅ Model loaded and working correctly!')

---
## ✅ Assignment Checklist

| Task | Status |
|---|---|
| Dataset loaded with Pandas | ✅ |
| `head()` displayed | ✅ |
| Dataset shape checked | ✅ |
| Missing values checked | ✅ |
| `describe()` statistics shown | ✅ |
| Additional EDA (distributions, heatmap, class balance) | ✅ |
| Features (X) and target (y) separated | ✅ |
| Train/Test split 80%/20% | ✅ |
| At least 2 models trained (5 trained) | ✅ |
| Accuracy, Precision, Recall, F1, ROC-AUC printed | ✅ |
| ROC Curve plotted | ✅ |
| ROC-AUC comparison across models | ✅ |
| Best model selected | ✅ |
| Model saved with `joblib` | ✅ |

---
*Module 11 — Diabetes Prediction Assignment*